# Ch 20 — llama.cpp 와 GGUF

HuggingFace 모델을 GGUF 포맷으로 변환하고 llama.cpp 로 노트북 추론을 띄웁니다.

In [ ]:
# 필요 시 설치
# !pip install torch transformers
# !pip install llama-cpp-python  # Python wrap

import subprocess
import os

print("llama.cpp 설치 여부 확인:")
result = subprocess.run(['which', 'llama-cli'], capture_output=True, text=True)
if result.returncode == 0:
    print(f"  llama-cli: {result.stdout.strip()}")
else:
    print("  llama-cli: 미설치 (아래 셀에서 설치)")

## 1. 개념 — GGUF 가 무엇인가

**GGUF (GPT-Generated Unified Format)** — `llama.cpp` 프로젝트가 정의한 단일 파일 모델 포맷.

```
[헤더]
  magic bytes "GGUF"
  metadata key-value pairs
    arch:           "llama" / "gpt2" / ...
    n_layer:        12
    vocab_size:     8000
    quantization:   "Q4_K_M"
[토크나이저]
  vocab + merges
[가중치]
  layer_0_attn_qkv  (int8 또는 int4 양자화)
  ...
```

### GGUF 양자화 변형

| 형식 | 비트 | 본 책 10M 크기 | PPL 손실 |
|---|---:|---:|---:|
| F16 | 16 | 20MB | 0% |
| Q8_0 | 8 | 10MB | <1% |
| Q5_K_M | 5 | 6MB | 1~2% |
| **Q4_K_M** | 4 | **5MB** | **3~5%** |
| Q3_K_S | 3 | 4MB | 8~15% |

→ **Q4_K_M 이 표준** — SmolLM2·Phi-3·Llama 3 모두 Q4_K_M 권장

## 4. 최소 예제 — HF → GGUF 변환

### 4.1 llama.cpp 설치

In [ ]:
# llama.cpp 빌드
# !git clone https://github.com/ggerganov/llama.cpp
# !cd llama.cpp && make -j4                    # CPU only (Colab / Linux)
# !cd llama.cpp && make GGML_METAL=1 -j4       # Apple Silicon
# !cd llama.cpp && make GGML_CUDA=1 -j4        # NVIDIA GPU
# !pip install -r llama.cpp/requirements.txt

print("llama.cpp 빌드 명령어:")
print("  Colab/Linux:     make -j4")
print("  Apple Silicon:   make GGML_METAL=1 -j4")
print("  NVIDIA GPU:      make GGML_CUDA=1 -j4")

In [ ]:
# convert.sh -- HF -> GGUF 변환

# 1. fp16 GGUF 변환
# !python llama.cpp/convert_hf_to_gguf.py \
#     runs/exp1/final \
#     --outfile dist/tiny-tale-f16.gguf \
#     --outtype f16

# 2. Q4_K_M 양자화 (fp16 GGUF -> Q4_K_M GGUF, 30초)
# !./llama.cpp/llama-quantize \
#     dist/tiny-tale-f16.gguf \
#     dist/tiny-tale-q4km.gguf \
#     Q4_K_M

# !ls -lh dist/
# tiny-tale-f16.gguf    20M
# tiny-tale-q4km.gguf    5M

print("변환 파이프라인:")
print("  HF transformers 형식 (save_pretrained)")
print("  -> convert_hf_to_gguf.py  -> fp16 GGUF")
print("  -> llama-quantize          -> Q4_K_M GGUF")

In [ ]:
# export_hf.py -- 본 책 모델을 HF 형식으로 export
# from transformers import GPT2Config, GPT2LMHeadModel
# from nano_gpt import GPTMini, GPTConfig
# import torch

# 1. 본 책 모델 로드
# cfg = GPTConfig(vocab_size=8000, n_layer=6, n_head=8, d_model=320, max_len=512)
# mine = GPTMini(cfg)
# mine.load_state_dict(torch.load("runs/exp1/final.pt")['model'])

# 2. GPT-2 형식 config 로 매핑 (가장 가까운 표준)
# hf_cfg = GPT2Config(
#     vocab_size=8000, n_layer=6, n_head=8, n_embd=320, n_positions=512,
#     activation_function="silu",   # SwiGLU 미지원 -- 가장 가까운 silu
# )
# hf = GPT2LMHeadModel(hf_cfg)

# 3. 가중치 매핑 (수동, 실제로는 30~50줄)
# ...

# hf.save_pretrained("runs/exp1/final_hf")
# tok.save_pretrained("runs/exp1/final_hf")   # tokenizer 도

print("주의:")
print("  본 책 GPTMini (RoPE + RMSNorm + SwiGLU)")
print("  vs GPT-2 (절대 PE + LayerNorm + GeLU) -- 완전 호환 X")
print()
print("권장 경로:")
print("  캡스톤에서 SmolLM2-360M 같이 이미 HF 호환되는 모델로 LoRA -> GGUF")
print("  본 책 본문: 자체 10M 모델은 PyTorch 까지만")

## 5. 실전 — llama-cli 로 띄우기

In [ ]:
# llama-cli 실행 (셸 명령)
print("CLI 실행 명령:")
cmd = """./llama.cpp/llama-cli \\
    -m dist/tiny-tale-q4km.gguf \\
    -p "Once upon a time" \\
    -n 100 \\
    --temp 0.8 \\
    --top-p 0.9 \\
    --no-display-prompt"""
print(cmd)
print()
print("전형적 출력:")
print("  Once upon a time, there was a little girl named Lily. She loved to play")
print("  with her teddy bear in the garden...")
print()
print("  llama_print_timings:        load time =     45.32 ms")
print("  llama_print_timings: prompt eval time =      8.12 ms /     5 tokens")
print("  llama_print_timings:        eval time =    234.56 ms /    99 runs")
print()
print("처리량: M2 MacBook 에서 본 책 10M Q4_K_M ~= 400 토큰/초")

In [ ]:
# llama_cpp_python.py -- Python wrap
# pip install llama-cpp-python

# from llama_cpp import Llama
# llm = Llama(model_path="dist/tiny-tale-q4km.gguf", n_ctx=512, verbose=False)
# out = llm("Once upon a time", max_tokens=100, temperature=0.8, top_p=0.9)
# print(out["choices"][0]["text"])

print("llama-cpp-python 사용 예시:")
print("  from llama_cpp import Llama")
print("  llm = Llama(model_path='dist/tiny-tale-q4km.gguf', n_ctx=512, verbose=False)")
print("  out = llm('Once upon a time', max_tokens=100, temperature=0.8, top_p=0.9)")
print("  print(out['choices'][0]['text'])")

## 연습문제

1. SmolLM2-360M 을 다운로드해 fp16 GGUF + Q4_K_M GGUF 두 가지로 변환. 파일 크기 비교.
2. 위 두 GGUF 로 `llama-cli` 같은 prompt 추론. 처리량(tok/s) + 출력 품질 차이는?
3. `convert_hf_to_gguf.py` 의 `--outtype` 을 `f16` / `bf16` / `q8_0` 으로 비교. 변환 시간 + 파일 크기.
4. 본 책 10M 모델을 (Llama-호환으로 재구현해서) GGUF 변환. Q4_K_M PPL 손실은?
5. **(생각해볼 것)** GGUF 가 PyTorch state_dict + safetensors 를 어떻게 대체하는가? HF 가 GGUF native 지원하는 의미는?

In [ ]:
# 연습 1: SmolLM2-360M 다운로드 + 변환


In [ ]:
# 연습 2: fp16 vs Q4_K_M 처리량 + 품질 비교


In [ ]:
# 연습 3: outtype 별 변환 시간 + 파일 크기


In [ ]:
# 연습 4: 본 책 10M 모델 GGUF 변환 + PPL


In [ ]:
# 연습 5: GGUF vs safetensors 구조 분석
